# AI 生態系統分析 — 描述性 EDA

本筆記本對兩個資料集進行探索性資料分析 (Exploratory Data Analysis)：

1. **arXiv AI/ML 論文** — 來自 Kaggle 的 AI 相關論文資料
2. **GitHub Trending Repos** — 來自 HuggingFace 的 GitHub 熱門專案月度排名

分析結果會輸出為 Tableau 可直接使用的 CSV 檔案，存放於 `data/processed/`。

---

## 環境設定

In [ ]:
import pandas as pd

from ai_ecosystem.analysis.descriptive import eda_arxiv, eda_github
from ai_ecosystem.ingest.arxiv import load_arxiv_data
from ai_ecosystem.ingest.github_trending import load_github_trending

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

---

## 1. arXiv 論文資料集

### 1.1 載入原始資料

資料來自 Kaggle，包含 AI 相關的 arXiv 論文 metadata。
預期約 7,700 篇論文，涵蓋 2025-09 到 2026-04 共 7 個月。

In [ ]:
arxiv_df = load_arxiv_data()
print(f"arXiv 資料集：{arxiv_df.shape[0]:,} 篇論文, {arxiv_df.shape[1]} 欄位")
arxiv_df.head(3)

### 1.2 執行 EDA

`eda_arxiv()` 會產生 6 張摘要表，分別用於 Tableau 的不同圖表：

In [ ]:
arxiv_tables = eda_arxiv(arxiv_df)
print("產出的表格：")
for name, table in arxiv_tables.items():
    print(f"  {name}: {len(table)} 列 × {len(table.columns)} 欄")

### 1.3 每月論文數量

觀察 AI 論文的時間趨勢。此表用於 Tableau 的折線圖。

In [ ]:
arxiv_tables["arxiv_monthly_volume"]

### 1.4 主要類別分布

arXiv 的 `primary_category` 欄位表示論文的主要分類（如 cs.CL、cs.CV、cs.LG 等）。
此表用於 Tableau 的長條圖或圓餅圖。

In [ ]:
arxiv_tables["arxiv_category_counts"].head(10)

### 1.5 類別 × 月份（堆疊面積圖用）

每個類別在每個月的論文數量，用於觀察各類別的成長趨勢。

In [ ]:
arxiv_tables["arxiv_category_monthly"].head(10)

### 1.6 最多產的作者（Top 20）

根據 `first_author` 欄位統計，顯示發表最多 AI 論文的前 20 位第一作者。

In [ ]:
arxiv_tables["arxiv_top_authors"]

### 1.7 摘要統計（每月平均字數）

追蹤每月論文摘要的平均長度，可用來觀察論文寫作風格的變化。

In [ ]:
arxiv_tables["arxiv_abstract_stats"]

### 1.8 類別共現分析

一篇論文可以標記多個類別（`all_categories`）。此表統計哪些類別經常同時出現，
可用於 Tableau 的熱力圖 (heatmap) 或網絡圖。

例如 cs.CL 與 cs.AI 的共現次數高，代表自然語言處理與人工智慧研究的高度重疊。

In [ ]:
arxiv_tables["arxiv_cross_category"].head(15)

---

## 2. GitHub Trending 資料集

### 2.1 載入原始資料

資料來自 HuggingFace，記錄 GitHub 每月 Top 25 熱門專案的排名。
預期約 3,200 列，涵蓋 2013-08 到 2025-11 共 128 個月。

> **注意**：star_count 和 fork_count 有約 32% 的缺失值，分析時會自動排除。

In [ ]:
github_df = load_github_trending()
print(f"GitHub 資料集：{github_df.shape[0]:,} 列, {github_df.shape[1]} 欄位")
github_df.head(3)

### 2.2 執行 EDA

`eda_github()` 會產生 6 張摘要表：

In [ ]:
github_tables = eda_github(github_df)
print("產出的表格：")
for name, table in github_tables.items():
    print(f"  {name}: {len(table)} 列 × {len(table.columns)} 欄")

### 2.3 每月上榜 Repo 數量

觀察 GitHub 熱門專案數量的時間趨勢。

In [ ]:
github_tables["github_monthly_entries"].head(10)

### 2.4 最常上榜的 Repos（Top 50）

根據 `ranking_appearances`（上榜次數）排序。
同時顯示平均排名、最新 star 數和 fork 數。

In [ ]:
github_tables["github_top_repos"].head(15)

### 2.5 AI/ML 相關 Repos 每月趨勢

使用關鍵字比對（如 ai, ml, llm, gpt, transformer 等）來標記 AI/ML 相關的 repo。
此表用於觀察 AI/ML 在 GitHub 熱門專案中的佔比變化。

In [ ]:
github_tables["github_aiml_monthly"].tail(12)

### 2.6 AI/ML 佔比（比率）

AI/ML repo 佔全部熱門 repo 的比例，按月份計算。
這是觀察 AI 熱潮的核心指標之一。

In [ ]:
github_tables["github_aiml_ratio"].tail(12)

### 2.7 Owner 集中度

哪些組織或個人擁有最多的熱門 repo？這反映了 GitHub 生態系統的集中程度。

In [ ]:
github_tables["github_owner_concentration"].head(15)

### 2.8 Star 成長軌跡

對於連續上榜 3 個月以上的 repo，計算其 star 數量的變化。
此表可用於 Tableau 的 slope chart 或 bump chart。

In [ ]:
github_tables["github_star_growth"].head(15)

---

## 3. 匯出 Tableau 用 CSV

執行以下程式碼，將所有摘要表匯出至 `data/processed/`。
之後即可在 Tableau 中直接匯入這些 CSV 檔案。

In [ ]:
from ai_ecosystem.analysis.export_tableau import export_all

exported = export_all()
print(f"\n已匯出 {len(exported)} 個 CSV 檔案：")
for p in exported:
    print(f"  {p}")

---

## 資料品質備註

| 項目 | 說明 |
|------|------|
| arXiv 時間範圍 | 僅 7 個月 (2025-09 ~ 2026-04)，趨勢分析有限 |
| GitHub star/fork 缺失 | 約 32% 為 null，star growth 分析僅包含有數據的 repo |
| AI/ML 標記 | 基於 repo 名稱關鍵字比對，可能遺漏某些 AI 專案 |
| GitHub 排名 | 每月僅 Top 25，無法觀察長尾分布 |

這些限制在 side project 的規模下是可接受的。詳細的資料品質報告請參見
`data/validation/` 目錄下的驗證報告。